<a href="https://colab.research.google.com/github/rameshaditya-me/Easy-Classical-ML-DL/blob/main/src/notebooks/discrete-regression/discrete-regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Discrete Regression
---

Discrete regression covers regression problems in which the **input**, the **target**, or both are **discrete** rather than continuous. Ordinary least squares is built for real-valued responses with Gaussian noise; once $x$ or $y$ takes values in a countable set, that setup no longer applies directly.

Given a dataset $\mathcal{D} = \{x_1, y_1, \ldots, x_n, y_n\}$, we may encounter settings such as:

- $x_i \in \mathbb{Z}^d$ and $y_i \in \mathbb{Z}^m$ (both discrete),
- $x_i \in \mathbb{R}^d$ and $y_i \in \mathbb{Z}^m$ (continuous input, discrete target),
- $x_i \in \mathbb{Z}^d$ and $y_i \in \mathbb{R}^m$ (discrete input, continuous target).

The main difficulty is that standard linear-regression assumptions — **normally distributed residuals** and **homoscedastic** (constant-variance) noise — often fail when variables are discrete. The appropriate model and loss then depend on whether the target is count-valued, ordinal, or categorical.

### Nature of Target Values
---

Discrete targets are not all handled the same way. Three common types are:

1. **Ordinal targets** carry a built-in ordering: one level is strictly higher or lower than another, but the numeric gap between levels need not mean anything. Examples include restaurant star ratings ($1$–$5$ stars) and medical pain scales (light, mild, severe, extreme). The model must respect order, yet treating the labels as arbitrary integers can be misleading.

2. **Categorical targets** partition the input space $\mathcal{X}$ into disjoint classes with **no natural order**. Classification is the canonical case: each $x_i$ is assigned to one of finitely many buckets, and swapping label indices should not change the problem.

3. **Count-valued targets** record frequencies or amounts (e.g. number of website visits, items sold). They take non-negative integers and often look like ordinal data, but **differences are meaningful**: ten events versus eleven is a quantifiable gap. By contrast, the step from $1$ to $2$ stars is ordered yet not an interval on a meaningful scale. This distinction separates counts from ordinals. 

### Modelling the problem using regression
---

#### Ordinal Targets

For ordinal targets, the model should respect the built-in ordering — for example $1 < 2 < 3 < 4 < 5$ stars. A naive starting point is to fit $K-1$ **independent** binary logistic models, one per cutpoint $k$, each modelling a cumulative probability:

$$
\phi_k(x) = P(y \leq k \mid x), \quad k = 1, \ldots, K-1
$$

At prediction time, evaluate $\phi_1(x), \ldots, \phi_{K-1}(x)$ and read off a rating — for instance, by thresholding each $\phi_k(x)$ at $0.5$ and decoding the resulting pattern of above/below decisions.

This approach has a clear weakness. Because the $K-1$ classifiers are trained **separately**, nothing guarantees that the cumulative probabilities are **monotone**:

$$
\phi_1(x) \leq \phi_2(x) \leq \cdots \leq \phi_{K-1}(x)
$$

Estimates can **contradict** one another (e.g. $\phi_2(x) < \phi_1(x)$), which makes the threshold rule **ambiguous** or unstable. The ordering among ranks must therefore be **built into the model**.

The **Frank–Hall** approach does this via a latent continuous score $Y^\star$. The observed rank is obtained by slicing $Y^\star$ with learned thresholds $\alpha_1, \ldots, \alpha_{K-1}$:

$$
Y = k \iff \alpha_{k-1} < Y^\star \leq \alpha_k
$$

(with $\alpha_0 = -\infty$ and $\alpha_K = +\infty$). Rather than fitting unrelated weight vectors $w_1, \ldots, w_{K-1}$, we use a **single** coefficient vector $\beta$ shared across cutpoints and model the cumulative distribution with logistic sigmoid:

$$
P(y \leq k \mid x) = \sigma(\alpha_k - \beta^\top x)
$$

Class probabilities are **differences** of adjacent cumulative terms:

$$
P(y = k \mid x) = \sigma(\alpha_k - \beta^\top x) - \sigma(\alpha_{k-1} - \beta^\top x)
$$

Expanding the first and last levels using the boundary conditions $\sigma(-\infty) = 0$ and $\sigma(+\infty) = 1$:

$$
P(y = 1 \mid x) = \sigma(\alpha_1 - \beta^\top x)
$$

$$
P(y = 2 \mid x) = \sigma(\alpha_2 - \beta^\top x) - \sigma(\alpha_1 - \beta^\top x)
$$

$$
P(y = 3 \mid x) = \sigma(\alpha_3 - \beta^\top x) - \sigma(\alpha_2 - \beta^\top x)
$$

$$
\vdots
$$

$$
P(y = K-1 \mid x) = \sigma(\alpha_{K-1} - \beta^\top x) - \sigma(\alpha_{K-2} - \beta^\top x)
$$

$$
P(y = K \mid x) = 1 - \sigma(\alpha_{K-1} - \beta^\top x)
$$

These $K$ probabilities partition the outcome space and sum to $1$ by construction, so rank ordering is enforced automatically.

### Categorical Targets

For categorical targets we can use the multinomial or softmax regression defined previously.


#### Count Based Targets

For count-based targets $y \in \mathbb{Z}_{\geq 0}$ (e.g. number of reviews, number of purchases), the natural distributional assumption is a **Poisson likelihood**:

$$P(y = k \mid \lambda) = \frac{e^{-\lambda} \lambda^k}{k!}$$

where $\lambda > 0$ is the rate parameter controlling both the mean and variance. A naive starting point is to fit a single $\lambda$ to the entire dataset, giving the combined likelihood:

$$L(\lambda) = \prod_{i=1}^{n} \frac{e^{-\lambda} \lambda^{y_i}}{y_i!} = \frac{e^{-n\lambda} \lambda^{\sum_{i=1}^{n} y_i}}{\prod_{i=1}^{n} y_i!}$$

This simply fits a Poisson distribution to the marginal counts, ignoring $x$ entirely. To make predictions conditional on $x$, we must let $\lambda$ vary per observation. Since $\lambda$ must remain strictly positive and $\beta^\top x$ is unconstrained on $\mathbb{R}$, we use the **exponential link**:

$$\lambda_i = e^{\beta^\top x_i}$$

Each observation now gets its own rate, and the log-likelihood becomes:

$$\ell(\beta) = \sum_{i=1}^{n} \left[ y_i \beta^\top x_i - e^{\beta^\top x_i} \right]$$

(constant $\log y_i!$ terms dropped). Maximising over $\beta$ gives the Poisson regression estimator.

At prediction time, $\hat{\lambda}_i = e^{\hat{\beta}^\top x_i}$ defines a full PMF over $\mathbb{Z}_{\geq 0}$. The choice of decoder depends on the loss being minimised:

$$\hat{y} = \begin{cases} \lfloor \hat{\lambda}_i \rfloor & \text{mode, minimises 0-1 loss} \\ \hat{\lambda}_i & \text{mean, minimises MSE} \\ \text{median}(\hat{\lambda}_i) & \text{median, minimises MAE} \end{cases}$$

---

### Generalized Linear Models

All of the models discussed — linear, logistic, softmax, ordered logit, and Poisson regression — are special cases of a single unifying framework: the **Generalized Linear Model (GLM)**. Every GLM is defined by three choices:

**1. A distribution** from the exponential family governing $y \mid x$

**2. A link function** $g(\cdot)$ that maps the linear predictor $\beta^\top x$ to the natural parameter of the distribution

**3. A loss** given by the negative log-likelihood of the chosen distribution, always optimised via MLE

The structural equation is always:

$$g\left(\mathbb{E}[y \mid x]\right) = \beta^\top x$$

The link function enforces the **support constraint** of the target — it ensures the predicted parameter lives in the valid range for that distribution:

| $y \mid x$ | Support | Distribution | Link $g$ | $g(\mathbb{E}[y\mid x]) = \beta^\top x$ |
|---|---|---|---|---|
| Continuous | $\mathbb{R}$ | Gaussian | Identity | $\mathbb{E}[y \mid x] = \beta^\top x$ |
| Binary | $\{0,1\}$ | Bernoulli | Logit | $\log \frac{p(x)}{1-p(x)} = \beta^\top x$ |
| Categorical | $\{1,\ldots,K\}$ unordered | Multinomial | Log | $\log \frac{p_k(x)}{p_K(x)} = \beta^\top x$ |
| Ordinal | $\{1,\ldots,K\}$ ordered | Cum. Logistic | Cum. Logit | $\log \frac{P(y \leq k \mid x)}{P(y > k \mid x)} = \alpha_k - \beta^\top x$ |
| Count | $\mathbb{Z}_{\geq 0}$ | Poisson | Log | $\log \lambda(x) = \beta^\top x$ |

The broader point is that **every entry in the last column is a function of $x$** — the whole purpose of the GLM framework is to model how the distribution of $y$ shifts as $x$ changes. 

The practical workflow for any supervised regression problem is therefore:

```
Observe the support of y|x
        ↓
Choose a distribution that respects that support
        ↓
Choose the canonical link for that distribution
        ↓
MLE gives β, thresholds, or both
        ↓
Decode predictions using the loss-appropriate statistic
       (mean / mode / median)
```

This framing makes explicit that **model selection is distribution selection** : the geometry of the output space determines everything else.

#### Interactive link-function demos

Each cell below fixes a **1D input** $x$ and shows how the conditional distribution $p(y \mid x)$ changes with $x$. The **left panel** is a 2D view over the $(x, y)$ plane (density, heatmap, or stacked probabilities); the **accent line** marks the slider value. The **right panel** is the distribution at that $x$, styled with **seaborn**. Move the slider to watch the link function shift mass across $y$.

In [ ]:
# 1. Gaussian — identity link: E[y|x] = β₀ + β₁x
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

sns.set_theme(
    style="whitegrid",
    context="talk",
    font_scale=0.82,
    rc={
        "axes.facecolor": "#f7f9fc",
        "figure.facecolor": "white",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.alpha": 0.35,
    },
)
ACCENT = "#e74c3c"
CMAP = sns.cubehelix_palette(start=0.15, rot=-0.2, dark=0.15, light=0.97, as_cmap=True)

beta0, beta1, sigma = 0.5, 1.2, 0.75
x_grid = np.linspace(-3, 3, 180)
X_MIN, X_MAX = float(x_grid[0]), float(x_grid[-1])
Y_MIN = beta0 + beta1 * X_MIN - 3.5 * sigma
Y_MAX = beta0 + beta1 * X_MAX + 3.5 * sigma
DENSITY_YMAX = 1.0 / (sigma * np.sqrt(2 * np.pi)) * 1.08
y_grid = np.linspace(Y_MIN, Y_MAX, 180)
X, Y = np.meshgrid(x_grid, y_grid)
MU = beta0 + beta1 * X
Z = np.exp(-0.5 * ((Y - MU) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))
mean_df = pd.DataFrame({"x": x_grid, "mean": beta0 + beta1 * x_grid})


def plot_gaussian_link(x0):
    mu0 = beta0 + beta1 * x0
    y_slice = np.linspace(Y_MIN, Y_MAX, 200)
    density = np.exp(-0.5 * ((y_slice - mu0) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))
    slice_df = pd.DataFrame({"y": y_slice, "density": density})

    fig, (ax_plane, ax_slice) = plt.subplots(
        1, 2, figsize=(12, 4.8), gridspec_kw={"width_ratios": [1.35, 1]}
    )
    fig.suptitle(
        r"Identity link: $\mathbb{E}[y \mid x] = \beta^{\top} x$",
        y=1.04, fontsize=13, weight="bold", color="#2c3e50",
    )

    ax_plane.contourf(X, Y, Z, levels=45, cmap=CMAP, alpha=0.95)
    sns.lineplot(
        data=mean_df, x="x", y="mean", ax=ax_plane,
        color="#2c3e50", lw=2.8, label=r"$\mathbb{E}[y \mid x] = \beta_0 + \beta_1 x$",
    )
    ax_plane.axvline(x0, color=ACCENT, lw=2.8, zorder=5, label=rf"slider: $x={x0:.2f}$")
    ax_plane.set_xlim(X_MIN, X_MAX)
    ax_plane.set_ylim(Y_MIN, Y_MAX)
    ax_plane.set_xlabel("$x$")
    ax_plane.set_ylabel("$y$")
    ax_plane.set_title("Conditional density over $(x,y)$")
    ax_plane.legend(loc="upper left", frameon=True, framealpha=0.92)

    sns.lineplot(data=slice_df, x="y", y="density", ax=ax_slice, color="#3498db", lw=2.6)
    ax_slice.fill_between(y_slice, density, alpha=0.28, color="#3498db")
    ax_slice.axvline(mu0, color="#7f8c8d", ls="--", lw=1.8, label=rf"mean $= {mu0:.2f}$")
    ax_slice.set_xlim(Y_MIN, Y_MAX)
    ax_slice.set_ylim(0, DENSITY_YMAX)
    ax_slice.set_xlabel("$y$")
    ax_slice.set_ylabel(r"$p(y \mid x)$")
    ax_slice.set_title(rf"Slice at $x = {x0:.2f}$")
    ax_slice.legend(frameon=True, framealpha=0.92)
    plt.tight_layout()
    plt.show()


interact(
    plot_gaussian_link,
    x0=FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.1, description="x", continuous_update=True),
)

In [ ]:
# 2. Bernoulli — logit link: log(p/(1-p)) = β₀ + β₁x
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

sns.set_theme(style="whitegrid", context="talk", font_scale=0.82, rc={"axes.facecolor": "#f7f9fc"})
ACCENT = "#e74c3c"
PALETTE = sns.color_palette("Set1", 2)


def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))


beta0, beta1 = -0.3, 1.4
x_grid = np.linspace(-3, 3, 180)
X_MIN, X_MAX = float(x_grid[0]), float(x_grid[-1])
p_grid = sigmoid(beta0 + beta1 * x_grid)
curve_df = pd.DataFrame({"x": x_grid, "p(y=1|x)": p_grid})
plane = np.vstack([1 - p_grid, p_grid])  # rows: y=0, y=1


def plot_bernoulli_link(x0):
    p1 = sigmoid(beta0 + beta1 * x0)
    slice_df = pd.DataFrame({"label": [r"$y=0$", r"$y=1$"], "p": [1 - p1, p1]})

    fig, (ax_plane, ax_slice) = plt.subplots(
        1, 2, figsize=(12, 4.8), gridspec_kw={"width_ratios": [1.35, 1]}
    )
    fig.suptitle(
        r"Logit link: $\log\!\left(\frac{p}{1-p}\right) = \beta^{\top} x$",
        y=1.04, fontsize=13, weight="bold", color="#2c3e50",
    )

    im = ax_plane.imshow(
        plane,
        aspect="auto",
        origin="lower",
        extent=[X_MIN, X_MAX, -0.5, 1.5],
        cmap=sns.color_palette("RdBu_r", as_cmap=True),
        vmin=0,
        vmax=1,
        interpolation="bilinear",
    )
    cbar = fig.colorbar(im, ax=ax_plane, shrink=0.85)
    cbar.set_label(r"$P(y \mid x)$")
    ax_plane.set_xlim(X_MIN, X_MAX)
    ax_plane.set_xlabel("$x$")
    ax_plane.set_yticks([0, 1])
    ax_plane.set_yticklabels([r"$y=0$", r"$y=1$"])
    ax_plane.set_title("Probability mass over $(x,y)$")

    ax2 = ax_plane.twinx()
    sns.lineplot(
        data=curve_df, x="x", y="p(y=1|x)", ax=ax2,
        color="#2c3e50", lw=2.6, label=r"$\sigma(\beta^{\top} x)$",
    )
    ax2.axvline(x0, color=ACCENT, lw=2.8, zorder=5)
    ax2.set_ylim(0, 1)
    ax2.set_ylabel(r"$p(y=1 \mid x)$")
    ax2.legend(loc="lower right", frameon=True, framealpha=0.92)

    sns.barplot(
        data=slice_df, x="label", y="p", hue="label", ax=ax_slice,
        palette=PALETTE, edgecolor="white", linewidth=1.4, dodge=False, legend=False,
    )
    for bar, val in zip(ax_slice.patches, slice_df["p"]):
        ax_slice.text(
            bar.get_x() + bar.get_width() / 2, val + 0.03,
            f"{val:.2f}", ha="center", va="bottom", fontsize=10, color="#2c3e50",
        )
    ax_slice.set_xlim(-0.6, 1.6)
    ax_slice.set_ylim(0, 1.12)
    ax_slice.set_ylabel(r"$P(y \mid x)$")
    ax_slice.set_title(rf"Slice at $x = {x0:.2f}$")
    plt.tight_layout()
    plt.show()


interact(
    plot_bernoulli_link,
    x0=FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.1, description="x", continuous_update=True),
)

In [ ]:
# 3. Multinomial — log link (softmax): log(p_k/p_K) = β_k^T x  (1D demo, K=3)
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

sns.set_theme(style="whitegrid", context="talk", font_scale=0.82, rc={"axes.facecolor": "#f7f9fc"})
ACCENT = "#e74c3c"
K = 3
CLASS_NAMES = ["A", "B", "C"]
COLORS = sns.color_palette("Set2", K)
b = np.array([1.2, 0.0, -1.0])
w = np.array([1.0, 0.2, -0.8])
x_grid = np.linspace(-3, 3, 180)
X_MIN, X_MAX = float(x_grid[0]), float(x_grid[-1])


def softmax_rows(scores):
    s = scores - scores.max(axis=1, keepdims=True)
    e = np.exp(np.clip(s, -500, 500))
    return e / e.sum(axis=1, keepdims=True)


scores = b + np.outer(x_grid, w)
probs = softmax_rows(scores)


def plot_multinomial_link(x0):
    eta = b + w * x0
    pk = np.exp(eta - eta.max())
    pk /= pk.sum()
    slice_df = pd.DataFrame({
        "label": [rf"$y={name}$" for name in CLASS_NAMES],
        "p": pk,
    })

    fig, (ax_plane, ax_slice) = plt.subplots(
        1, 2, figsize=(12, 4.8), gridspec_kw={"width_ratios": [1.35, 1]}
    )
    fig.suptitle(
        r"Log link (softmax): $\log\!\left(\frac{p_k}{p_K}\right) = \beta_k^{\top} x$",
        y=1.04, fontsize=13, weight="bold", color="#2c3e50",
    )

    ax_plane.stackplot(
        x_grid, probs.T,
        labels=[rf"$y={name}$" for name in CLASS_NAMES],
        colors=COLORS, alpha=0.88,
    )
    for k, name in enumerate(CLASS_NAMES):
        ax_plane.plot(x_grid, probs[:, k], color=COLORS[k], lw=1.2, alpha=0.9)
    ax_plane.axvline(x0, color=ACCENT, lw=2.8, zorder=6, label=rf"slider: $x={x0:.2f}$")
    ax_plane.set_xlim(X_MIN, X_MAX)
    ax_plane.set_ylim(0, 1)
    ax_plane.set_xlabel("$x$")
    ax_plane.set_ylabel(r"$P(y \mid x)$")
    ax_plane.set_title("Stacked class probabilities vs $x$")
    ax_plane.legend(loc="upper left", frameon=True, framealpha=0.92, ncol=2)

    sns.barplot(
        data=slice_df, x="label", y="p", hue="label", ax=ax_slice,
        palette=COLORS, edgecolor="white", linewidth=1.4, dodge=False, legend=False,
    )
    for bar, val in zip(ax_slice.patches, pk):
        ax_slice.text(
            bar.get_x() + bar.get_width() / 2, val + 0.02,
            f"{val:.2f}", ha="center", va="bottom", fontsize=10,
        )
    ax_slice.set_xlim(-0.6, K - 0.4)
    ax_slice.set_ylim(0, 1.1)
    ax_slice.set_ylabel(r"$P(y \mid x)$")
    ax_slice.set_title(rf"Slice at $x = {x0:.2f}$")
    plt.tight_layout()
    plt.show()


interact(
    plot_multinomial_link,
    x0=FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.1, description="x", continuous_update=True),
)

In [ ]:
# 4. Ordinal — cumulative logit: log(P(y≤k)/P(y>k)) = α_k − β^T x  (K=5)
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

sns.set_theme(style="whitegrid", context="talk", font_scale=0.82, rc={"axes.facecolor": "#f7f9fc"})
ACCENT = "#e74c3c"


def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))


K = 5
alphas = np.array([-2.2, -0.7, 0.7, 2.2])
beta = 1.0
x_grid = np.linspace(-3, 3, 180)
X_MIN, X_MAX = float(x_grid[0]), float(x_grid[-1])
COLORS = sns.color_palette("RdYlGn", K)


def ordinal_probs(x):
    cum = sigmoid(alphas - beta * x)
    edges = np.concatenate([[0.0], cum, [1.0]])
    return np.diff(edges)


prob_grid = np.array([ordinal_probs(x) for x in x_grid])
SLICE_YMAX = prob_grid.max() + 0.14


def plot_ordinal_link(x0):
    pk = ordinal_probs(x0)
    slice_df = pd.DataFrame({
        "label": [rf"$y={k}$" for k in range(1, K + 1)],
        "p": pk,
    })

    fig, (ax_plane, ax_slice) = plt.subplots(
        1, 2, figsize=(12, 4.8), gridspec_kw={"width_ratios": [1.35, 1]}
    )
    fig.suptitle(
        r"Cumulative logit: $\log\!\left(\frac{P(y \leq k)}{P(y > k)}\right) = \alpha_k - \beta^{\top} x$",
        y=1.04, fontsize=13, weight="bold", color="#2c3e50",
    )

    ax_plane.stackplot(x_grid, prob_grid.T, colors=COLORS, alpha=0.9)
    for k in range(K):
        ax_plane.plot(x_grid, prob_grid[:, k], color=COLORS[k], lw=1.0, alpha=0.85)
    ax_plane.axvline(x0, color=ACCENT, lw=2.8, zorder=6)
    ax_plane.set_xlim(X_MIN, X_MAX)
    ax_plane.set_ylim(0, 1)
    ax_plane.set_xlabel("$x$")
    ax_plane.set_ylabel(r"$P(y=k \mid x)$")
    ax_plane.set_title("Ordinal rank probabilities vs $x$")

    sns.barplot(
        data=slice_df, x="label", y="p", hue="label", ax=ax_slice,
        palette=COLORS, edgecolor="white", linewidth=1.4, dodge=False, legend=False,
    )
    for bar, val in zip(ax_slice.patches, pk):
        ax_slice.text(
            bar.get_x() + bar.get_width() / 2, val + 0.015,
            f"{val:.2f}", ha="center", va="bottom", fontsize=10,
        )
    ax_slice.set_xlim(-0.6, K - 0.4)
    ax_slice.set_ylim(0, SLICE_YMAX)
    ax_slice.set_ylabel(r"$P(y=k \mid x)$")
    ax_slice.set_title(rf"Slice at $x = {x0:.2f}$")
    plt.tight_layout()
    plt.show()


interact(
    plot_ordinal_link,
    x0=FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.1, description="x", continuous_update=True),
)

In [ ]:
# 5. Poisson — log link: log λ(x) = β₀ + β₁x
import math
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

sns.set_theme(style="whitegrid", context="talk", font_scale=0.82, rc={"axes.facecolor": "#f7f9fc"})
ACCENT = "#e74c3c"
CMAP = sns.color_palette("mako", as_cmap=True)

beta0, beta1 = 0.2, 0.9
x_grid = np.linspace(-3, 3, 180)
X_MIN, X_MAX = float(x_grid[0]), float(x_grid[-1])
lam_grid = np.exp(beta0 + beta1 * x_grid)
rate_df = pd.DataFrame({"x": x_grid, "lambda": lam_grid})
k_max = 18
k_vals = np.arange(0, k_max + 1)
LAM_MAX = float(np.exp(beta0 + beta1 * X_MAX))
K_SLICE_MAX = min(k_max, int(LAM_MAX + 5 * np.sqrt(max(LAM_MAX, 0.2))) + 1)
K_SLICE = np.arange(0, K_SLICE_MAX + 1)


def poisson_pmf(k, lam):
    k = np.asarray(k, dtype=float)
    log_p = k * np.log(lam + 1e-12) - lam - np.vectorize(lambda kk: math.lgamma(kk + 1.0))(k)
    return np.exp(log_p)


SLICE_PROB_YMAX = max(
    float(poisson_pmf(K_SLICE, lam).max()) for lam in lam_grid
) * 1.08
plane = np.array([poisson_pmf(k_vals, lam) for lam in lam_grid]).T  # rows = counts


def plot_poisson_link(x0):
    lam0 = np.exp(beta0 + beta1 * x0)
    pk = poisson_pmf(K_SLICE, lam0)
    slice_df = pd.DataFrame({"count": K_SLICE, "p": pk})

    fig, (ax_plane, ax_slice) = plt.subplots(
        1, 2, figsize=(12, 4.8), gridspec_kw={"width_ratios": [1.35, 1]}
    )
    fig.suptitle(
        r"Log link: $\log \lambda(x) = \beta^{\top} x$",
        y=1.04, fontsize=13, weight="bold", color="#2c3e50",
    )

    im = ax_plane.imshow(
        plane,
        aspect="auto",
        origin="lower",
        extent=[X_MIN, X_MAX, -0.5, k_max + 0.5],
        cmap=CMAP,
        vmin=0,
        vmax=plane.max(),
        interpolation="bilinear",
    )
    cbar = fig.colorbar(im, ax=ax_plane, shrink=0.85)
    cbar.set_label(r"$P(y \mid x)$")
    ax_plane.set_xlim(X_MIN, X_MAX)
    ax_plane.set_xlabel("$x$")
    ax_plane.set_ylabel("count $y$")
    ax_plane.set_title("Poisson PMF mass over $(x,y)$")

    ax2 = ax_plane.twinx()
    sns.lineplot(
        data=rate_df, x="x", y="lambda", ax=ax2,
        color=ACCENT, lw=2.6, label=r"$\lambda(x)=e^{\beta^{\top} x}$",
    )
    ax2.axvline(x0, color="#2c3e50", lw=2.2, ls="--", zorder=5)
    ax2.set_ylabel(r"$\lambda(x)$")
    ax2.legend(loc="upper left", frameon=True, framealpha=0.92)

    sns.barplot(
        data=slice_df, x="count", y="p", color="#7e57c2", ax=ax_slice,
        edgecolor="white", linewidth=1.2,
    )
    ax_slice.axvline(lam0, color="#7f8c8d", ls="--", lw=1.8, label=rf"$\lambda = {lam0:.2f}$")
    ax_slice.set_xlim(-0.6, K_SLICE_MAX + 0.6)
    ax_slice.set_ylim(0, SLICE_PROB_YMAX)
    ax_slice.set_xlabel("$y$")
    ax_slice.set_ylabel(r"$P(y \mid x)$")
    ax_slice.set_title(rf"Slice at $x = {x0:.2f}$")
    ax_slice.legend(frameon=True, framealpha=0.92)
    plt.tight_layout()
    plt.show()


interact(
    plot_poisson_link,
    x0=FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.1, description="x", continuous_update=True),
)